In [1]:
import os
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN")
login(hf_token)

if hf_token:
    login(hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("Hugging Face token not found. Add it in Colab Secrets.")


Hugging Face token not found. Add it in Colab Secrets.


In [2]:
# Install specific compatible versions
import sys

!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install numpy scikit-learn


!{sys.executable} -m pip install git+https://github.com/AIPHES/emnlp19-moverscore.git

!{sys.executable} -m pip install accelerate rouge-score bert-score nltk torch torchvision pyemd datasets transformers peft "torchao>=0.16.0"

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

# Verify
try:
    from moverscore_v2 import get_idf_dict, word_mover_score
    print("✓ MoverScore installed successfully")
except Exception as e:
    print(f"✗ Error: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.8 MB/s  0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.12.0+cu130 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 141.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 176.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn] [scikit-learn]
  Cloning https://github.com/AIPHES/emnlp19-moverscore.git to ./tmp/pip-req-build-0du480zc
  Running command git clone --filter=blob:none --quiet https://github.com/AIPHES/emnlp19-moverscore.git /tmp/pip-req-build-0du480zc
  Resolved https://github.com/

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

W0726 11:17:28.468000 1414 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0726 11:17:28.492000 1414 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ MoverScore installed successfully


In [3]:
import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)

import torch
import gc
import math
import os
import threading
from typing import Optional
from PIL import Image
from io import BytesIO
import requests
from huggingface_hub import hf_hub_download


# Patch for numpy.float deprecation in MoverScore
import numpy as np
if not hasattr(np, 'float'):
    np.float = float

import re
import numpy as np
from transformers import EvalPrediction
import nltk
nltk.download('punkt', quiet=True)

from datasets import load_dataset
from transformers import (
    Gemma3ForConditionalGeneration,
    AutoProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from moverscore_v2 import get_idf_dict, word_mover_score
from peft import AdaLoraConfig, get_peft_model


class LazyImageCache:
    """Thread-safe cache for loaded images to avoid re-downloading."""
    def __init__(self, max_size: int = 100):
        self.cache = {}
        self.max_size = max_size
        self.lock = threading.Lock()

    def get(self, key: str) -> Optional[Image.Image]:
        with self.lock:
            return self.cache.get(key)

    def put(self, key: str, image: Image.Image):
        with self.lock:
            if len(self.cache) >= self.max_size:
                oldest_key = next(iter(self.cache))
                del self.cache[oldest_key]
            self.cache[key] = image

    def clear(self):
        with self.lock:
            self.cache.clear()


_image_cache = LazyImageCache()


def load_image_lazy(image_filename: str,
                   repo_id: str = "bitmind/MS-COCO",
                   cache_dir: str = "./IC_images_cache") -> Image.Image:
    """Load image on-demand with caching to avoid repeated downloads."""
    cached_img = _image_cache.get(image_filename)
    if cached_img is not None:
        return cached_img.copy()

    if not isinstance(image_filename, str):
        if hasattr(image_filename, 'convert'):
            return image_filename.convert('RGB')
        else:
            placeholder = Image.new('RGB', (224, 224), (128, 128, 128))
            return placeholder

    filename = image_filename.strip()
    if not filename:
        return Image.new('RGB', (224, 224), (128, 128, 128))

    if os.path.exists(filename):
        try:
            img = Image.open(filename).convert('RGB')
            _image_cache.put(filename, img)
            return img.copy()
        except Exception:
            pass

    folders_to_try = [f"images{i}" for i in range(1, 8)]
    for folder in folders_to_try:
        try:
            file_path = f"{folder}/{filename}"
            local_path = hf_hub_download(
                repo_id=repo_id,
                filename=file_path,
                cache_dir=cache_dir,
                repo_type="dataset"
            )

            img = Image.open(local_path)
            img.verify()
            img = Image.open(local_path).convert('RGB')
            _image_cache.put(filename, img)
            return img.copy()
        except Exception:
            continue

    print(f"[WARN] Could not find image {filename}, using placeholder")
    placeholder = Image.new('RGB', (224, 224), (128, 128, 128))
    return placeholder


def load_image_from_source(image_source):
    """
    Robust image loading that handles strings, PIL Images, URLs, and file paths.

    Args:
        image_source: Can be a PIL Image, file path string, URL string, or base64 string

    Returns:
        PIL.Image in RGB mode or None if loading fails
    """
    try:
        if isinstance(image_source, Image.Image):
            return image_source.convert('RGB') if image_source.mode != 'RGB' else image_source

        if isinstance(image_source, str):
            if image_source.startswith(('http://', 'https://')):
                response = requests.get(image_source, timeout=10)
                response.raise_for_status()
                image = Image.open(BytesIO(response.content))
                return image.convert('RGB') if image.mode != 'RGB' else image

            if os.path.exists(image_source):
                image = Image.open(image_source)
                return image.convert('RGB') if image.mode != 'RGB' else image

            try:
                import base64
                image_data = base64.b64decode(image_source)
                image = Image.open(BytesIO(image_data))
                return image.convert('RGB') if image.mode != 'RGB' else image
            except Exception:
                pass

        if isinstance(image_source, bytes):
            image = Image.open(BytesIO(image_source))
            return image.convert('RGB') if image.mode != 'RGB' else image

        return None

    except Exception as e:
        print(f"Failed to load image: {e}")
        return None


class AdaLoRAConfig:
    """Configuration for LoRA-only training on Gemma 3-4B-IT models."""

    def __init__(self):
        # Gemma 3-4B-IT model ID
        self.model_id = "google/gemma-3-4b-it"
        self.batch_size = 1
        self.gradient_accumulation_steps = 4
        self.num_epochs = 3
        self.learning_rate = 2e-6
        self.precision = "bf16"
        self.max_length = 512
        self.max_caption_length = 128

        # LoRA configuration
        self.lora_rank = 8
        self.lora_alpha = 16
        self.lora_dropout = 0.05

        # Gemma 3 target modules
        self.lora_target_modules = [
            "q_proj",
            "k_proj",
            "v_proj"
        ]

        # Vision encoder LoRA
        self.apply_lora_to_vision = True
        self.vision_lora_modules = [
            "q_proj",
            "k_proj",
            "v_proj"
        ]

        # Dataset configuration
        self.dataset_name = "bitmind/MS-COCO"
        self.num_workers = 0
        self.pin_memory = True
        self.drop_last = True

        self.answer_max_length = 512
        self.ignore_index = -100


def main():
    config = AdaLoRAConfig()

    print("Loading processor and model for Gemma 3-4B-IT...")
    processor = AutoProcessor.from_pretrained(
        config.model_id,
        token=hf_token,
        trust_remote_code=True,
        padding_side="left",
    )

    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token



    model = Gemma3ForConditionalGeneration.from_pretrained(
        config.model_id,
        device_map="auto",
        token=hf_token,
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )

    # Apply LoRA configuration
    target_modules = config.lora_target_modules.copy()

    lora_config = AdaLoraConfig(
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        target_modules=target_modules,
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        inference_mode=False,
        use_rslora=True,
        total_step=24
    )

    model = get_peft_model(model, lora_config)

    # Count vision and language modules
    from peft.tuners.lora import LoraLayer
    vision_modules = 0
    language_modules = 0

    for name, module in model.named_modules():
        if isinstance(module, LoraLayer) and module.r.get('default', 0) > 0:
            name_lower = name.lower()
            is_vision = any([
                'vision_tower' in name_lower,
                'vision_model' in name_lower,
                'vision_encoder' in name_lower,
                'visual' in name_lower and 'language' not in name_lower,
                name.startswith('model.vision'),
                name.startswith('vision_tower'),
            ])

            if is_vision:
                vision_modules += 1
            else:
                language_modules += 1

    print(f"Instrumented {vision_modules} vision modules + {language_modules} language modules")
    print(f"Total LoRA modules: {vision_modules + language_modules}")

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"Trainable params: {trainable_params:,}")
    print(f"Total params: {total_params:,}")
    print(f"Percentage: {100 * trainable_params / total_params:.3f}%")

    model.enable_input_require_grads()
    model.config.use_cache = False

    print("Loading VQAonline dataset with splits...")
    train_ds = load_dataset(config.dataset_name, split="train", streaming=True, cache_dir="./ic_cache")
    val_ds = load_dataset(config.dataset_name, split="validation", streaming=True, cache_dir="./ic_cache")
    test_ds = load_dataset(config.dataset_name, split="test", streaming=True, cache_dir="./ic_cache")

    train_dataset = train_ds.take(665)
    val_dataset = val_ds.take(83)
    test_dataset = test_ds.take(83)

    #print(f"Final splits → Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

    def tokenize_vqa(example):
        """Process VQA examples with robust image loading for Gemma 3."""
        try:
            if 'image' not in example or example['image'] is None:
                return None

            image = example["image"]

            if image.mode != "RGB":
                image = image.convert("RGB")  #change

            if image is None:
                return None

            if not isinstance(image, Image.Image):
                return None
            if image.mode != 'RGB':
                image = image.convert('RGB')

            caption = example["sentences"]["raw"].strip()

            if not caption:
                return None

            # if not question or not answer:
            #     return None

            text_content = (
                "Describe this image in one sentence. "
                "Output only the caption."
            )

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": text_content}
                    ]
                }
            ]

            text_prompt = processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            inputs = processor(
                text=[text_prompt],
                images=[image],
                return_tensors="pt",
                padding="max_length",
                max_length=config.max_length,
                truncation=True,
            )

            caption_tokens = processor.tokenizer(
                caption,
                add_special_tokens=False,
                max_length=config.max_caption_length,
                truncation=True,
                return_tensors="pt"
            )["input_ids"].squeeze(0)

            input_ids = inputs['input_ids'].squeeze(0)
            attention_mask = inputs['attention_mask'].squeeze(0)

            if 'token_type_ids' not in inputs or inputs['token_type_ids'] is None:
                inputs['token_type_ids'] = torch.zeros_like(inputs['input_ids'])
            token_type_ids = inputs['token_type_ids'].squeeze(0)

            if not isinstance(input_ids, torch.Tensor):
                input_ids = torch.tensor(input_ids, dtype=torch.long)
            if not isinstance(attention_mask, torch.Tensor):
                attention_mask = torch.tensor(attention_mask, dtype=torch.long)
            if not isinstance(caption_tokens, torch.Tensor):
                answer_tokens = torch.tensor(caption_tokens, dtype=torch.long)
            if token_type_ids is not None and not isinstance(token_type_ids, torch.Tensor):
                token_type_ids = torch.tensor(token_type_ids, dtype=torch.long)

            labels = input_ids.clone()
            labels[:] = config.ignore_index

            caption_len = min(len(caption_tokens), len(labels))

            if caption_len > 0:
                labels[-caption_len:] = caption_tokens[:caption_len]

            output = {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'token_type_ids': token_type_ids,
                'labels': labels
            }

            if 'pixel_values' in inputs:
                pixel_values = inputs['pixel_values']

                if isinstance(pixel_values, torch.Tensor):
                    if pixel_values.dim() >= 2 and pixel_values.shape[0] == 1:
                        pixel_values = pixel_values.squeeze(0)
                    output['pixel_values'] = pixel_values
                elif isinstance(pixel_values, list) and len(pixel_values) > 0:
                    if isinstance(pixel_values[0], torch.Tensor):
                        output['pixel_values'] = pixel_values[0] if len(pixel_values) == 1 else torch.cat(pixel_values, dim=0)

            for key in ['input_ids', 'attention_mask', 'token_type_ids', 'labels']:
                if key in output and not isinstance(output[key], torch.Tensor):
                    print(f"ERROR: {key} is not a tensor: {type(output[key])}")
                    return None

            return output

        except Exception as e:
            print(f"Error processing example: {e}")
            import traceback
            traceback.print_exc()
            return None

    print("Tokenizing datasets...")
    print("Processing training data...")
    tokenized_train_dataset = train_dataset.map(
        tokenize_vqa,
        remove_columns=train_dataset.column_names,
        #num_proc=None,
        #desc="Processing training data",
        #load_from_cache_file=False
    )

    print("Filtering valid training samples...")
    tokenized_train_dataset = tokenized_train_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        #desc="Filtering training data"
    )

    print("Processing validation data...")
    tokenized_val_dataset = val_dataset.map(
        tokenize_vqa,
        remove_columns=val_dataset.column_names,
        #num_proc=None,
        #desc="Processing validation data",
        #load_from_cache_file=False
    )

    print("Filtering valid validation samples...")
    tokenized_val_dataset = tokenized_val_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        #desc="Filtering validation data"
    )

    print("Processing test data...")
    tokenized_test_dataset = test_dataset.map(
        tokenize_vqa,
        remove_columns=test_dataset.column_names,
        #num_proc=None,
        #desc="Tokenizing test",
        #load_from_cache_file=False,
    )

    print("Filtering valid test samples...")
    tokenized_test_dataset = tokenized_test_dataset.filter(
        lambda x: x is not None and 'input_ids' in x and 'labels' in x,
        #desc="Filtering test data"
    )
    from datasets import Dataset

    tokenized_train_dataset = Dataset.from_list(list(tokenized_train_dataset))
    tokenized_val_dataset = Dataset.from_list(list(tokenized_val_dataset))
    tokenized_test_dataset = Dataset.from_list(list(tokenized_test_dataset))

    train_len = 32
    val_len = 4
    test_len = 4

    if train_len == 0:
        raise ValueError(
            "No training samples after processing. "
            "Possible causes: images not loading correctly, dataset format incompatible, "
            "or all examples failed validation."
        )

    print(f"\nSuccessfully processed datasets:")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")

    eff_batch = config.batch_size * config.gradient_accumulation_steps
    total_steps = math.ceil(train_len / eff_batch) * config.num_epochs

    print(f"\nTraining Configuration:")
    print(f"Total training steps: {total_steps}")
    print(f"Effective batch size: {eff_batch}")
    print(f"Steps per epoch: {math.ceil(train_len / eff_batch)}")

    training_args = Seq2SeqTrainingArguments(
        output_dir="./results_vqa_lora_only_gemma3",
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        num_train_epochs=config.num_epochs,
        learning_rate=config.learning_rate,
        bf16=(config.precision == "bf16"),
        fp16=(config.precision == "fp16"),
        max_grad_norm=1.5,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataloader_num_workers=config.num_workers,
        dataloader_pin_memory=config.pin_memory,
        dataloader_drop_last=config.drop_last,
        remove_unused_columns=False,
        report_to="none",
        save_strategy="steps",
        eval_steps=8,
        save_steps=8,
        eval_strategy="steps",
        save_total_limit=2,
        logging_steps=50,
        warmup_steps=100,
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model="loss",
        prediction_loss_only=False,  # Need full predictions for metrics
        predict_with_generate=True,  # Use generation for evaluation
        generation_max_length=config.max_length,
        max_steps=24,
    )

    def collate_fn(batch):
        """Custom collator for Gemma 3 inputs with proper batching."""
        batch = [b for b in batch if b is not None and isinstance(b, dict) and 'input_ids' in b]

        if len(batch) == 0:
            print("WARNING: Empty batch after filtering - creating dummy batch")
            dummy_input_ids = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_attention_mask = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_labels = torch.full((1, config.max_length), config.ignore_index, dtype=torch.long)
            dummy_token_type_ids = torch.zeros((1, config.max_length), dtype=torch.long) # New

            return {
                'input_ids': dummy_input_ids,
                'attention_mask': dummy_attention_mask,
                'labels': dummy_labels,
                'token_type_ids': dummy_token_type_ids, # New
            }

        try:
            for i, item in enumerate(batch):
                for key in ['input_ids', 'attention_mask', 'labels', 'token_type_ids']:
                    if key in item and not isinstance(item[key], torch.Tensor):
                        batch[i][key] = torch.tensor(item[key], dtype=torch.long)

            input_ids = torch.nn.utils.rnn.pad_sequence(
                [b['input_ids'] for b in batch],
                batch_first=True,
                padding_value=processor.tokenizer.pad_token_id
            )

            attention_mask = torch.nn.utils.rnn.pad_sequence(
                [b['attention_mask'] for b in batch],
                batch_first=True,
                padding_value=0
            )

            labels = torch.nn.utils.rnn.pad_sequence(
                [b['labels'] for b in batch],
                batch_first=True,
                padding_value=config.ignore_index
            )

            output = {
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': labels,
            }

            if 'token_type_ids' in batch[0]:
                token_type_ids_list = [
                    b['token_type_ids'] for b in batch
                    if 'token_type_ids' in b and isinstance(b['token_type_ids'], torch.Tensor)  # ← ensure tensor
                ]
                if token_type_ids_list:
                    token_type_ids = torch.nn.utils.rnn.pad_sequence(
                        token_type_ids_list,
                        batch_first=True,
                        padding_value=0
                    )
                    output['token_type_ids'] = token_type_ids

            if 'pixel_values' in batch[0] and batch[0]['pixel_values'] is not None:
                pixel_values_list = []

                for b in batch:
                    pv = b['pixel_values']

                    if isinstance(pv, list):
                        if len(pv) > 0 and isinstance(pv[0], torch.Tensor):
                            pv = torch.stack(pv) if len(pv) > 1 else pv[0]
                        elif len(pv) > 0 and isinstance(pv[0], list):
                            pv = torch.tensor(pv)
                        else:
                            continue

                    if isinstance(pv, torch.Tensor):
                        while pv.dim() > 0 and pv.shape[0] == 1:
                            pv = pv.squeeze(0)
                        pixel_values_list.append(pv)

                if len(pixel_values_list) > 0:
                    try:
                        output['pixel_values'] = torch.stack(pixel_values_list, dim=0)
                    except RuntimeError:
                        max_shapes = [max(pv.shape[i] for pv in pixel_values_list) for i in range(pixel_values_list[0].dim())]
                        padded_pvs = []
                        for pv in pixel_values_list:
                            if list(pv.shape) != max_shapes:
                                padding = []
                                for i in range(len(max_shapes) - 1, -1, -1):
                                    padding.extend([0, max_shapes[i] - pv.shape[i]])
                                pv = torch.nn.functional.pad(pv, padding)
                            padded_pvs.append(pv)
                        output['pixel_values'] = torch.stack(padded_pvs, dim=0)

            return output

        except Exception as e:
            print(f"ERROR in collate_fn: {e}")
            import traceback
            traceback.print_exc()

            dummy_input_ids = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_attention_mask = torch.zeros((1, config.max_length), dtype=torch.long)
            dummy_labels = torch.full((1, config.max_length), config.ignore_index, dtype=torch.long)
            dummy_token_type_ids = torch.zeros((1, config.max_length), dtype=torch.long) # New

            return {
                'input_ids': dummy_input_ids,
                'attention_mask': dummy_attention_mask,
                'labels': dummy_labels,
                'token_type_ids': dummy_token_type_ids, # New
            }

    def make_compute_metrics(processor):
        import re
        import numpy as np
        from rouge_score import rouge_scorer
        from nltk.translate.meteor_score import meteor_score
        from bert_score import score as bert_score
        from collections import defaultdict
        import nltk
        nltk.download('wordnet', quiet=True);
        nltk.download('omw-1.4', quiet=True)

        # PATCH FOR MOVERSCORE COMPATIBILITY
        import transformers.tokenization_utils_base as tub
        if not hasattr(tub.PreTrainedTokenizerBase, '_moverscore_patched'):
            original_init = tub.PreTrainedTokenizerBase.__init__
            def patched_init(self, *args, **kwargs):
                original_init(self, *args, **kwargs)
                if not hasattr(self, 'max_len'):
                    self.max_len = self.model_max_length
            tub.PreTrainedTokenizerBase.__init__ = patched_init
            tub.PreTrainedTokenizerBase._moverscore_patched = True
            print("✓ Applied moverscore compatibility patch")

        # Now import moverscore
        moverscore_available = False
        try:
            from moverscore_v2 import get_idf_dict, word_mover_score
            moverscore_available = True
            print("✓ MoverScore loaded successfully")
        except ImportError as e:
            print(f"✗ MoverScore not available: {e}")

        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)


        def extract_answer(text):
            """Clean Gemma output safely without truncating content."""

            if text is None:
                return ""

            # Remove special tokens
            text = text.replace("<bos>", "")
            text = text.replace("<eos>", "")
            text = text.replace("<start_of_turn>", "")
            text = text.replace("<end_of_turn>", "")

            # Remove leading 'model' tag if present
            text = re.sub(r"^\s*model\s*", "", text, flags=re.IGNORECASE)

            # Normalize whitespace
            text = re.sub(r"\s+", " ", text).strip()

            return text

        def normalize(s):
            """VQAonline normalization."""
            s = re.sub(r"\s+", " ", re.sub(r"([.!?,;])", r" \1", s.lower().strip()))
            s = re.sub(r" n't ", " not ", s)
            s = re.sub(r" 's ", " is ", s)
            s = re.sub(r" 're ", " are ", s)
            s = re.sub(r" 've ", " have ", s)
            s = re.sub(r" 'll ", " will ", s)
            s = re.sub(r" 'd ", " would ", s)
            return s.strip()

        def compute_vqa_metrics(p: EvalPrediction):
            pred_ids = np.asarray(p.predictions, dtype=np.int64)
            labels = np.asarray(p.label_ids, dtype=np.int64)

            vocab_size = processor.tokenizer.vocab_size
            pad_token_id = processor.tokenizer.pad_token_id

            pred_ids = np.clip(pred_ids, 0, vocab_size - 1)
            labels[labels < 0] = pad_token_id
            labels = np.clip(labels, 0, vocab_size - 1)

            try:
                pred_texts_raw = processor.batch_decode(pred_ids, skip_special_tokens=True)
                label_texts = processor.batch_decode(labels, skip_special_tokens=True)
            except Exception as e:
                print(f"[Metrics] Decode error: {e}")
                return {"rouge_l": 0.0, "meteor": 0.0, "bertscore_f1": 0.0, "moverscore": 0.0}

            pred_texts = [extract_answer(t) for t in pred_texts_raw]
            pred_norm = [normalize(t) for t in pred_texts]
            label_norm = [normalize(t) for t in label_texts]

            for i in range(min(3, len(pred_norm))):
                print(f"\n[Sample {i+1}]")
                print(f"Generated: {pred_texts[i][:100]}")
                print(f"Reference: {label_texts[i][:100]}")

            valid_preds = []
            valid_labels = []
            for p, l in zip(pred_norm, label_norm):
                if p and l and len(p.split()) > 0 and len(l.split()) > 0:
                    valid_preds.append(p)
                    valid_labels.append(l)

            # ROUGE-L
            rouge_ls = []
            for p, l in zip(pred_norm, label_norm):
                if p and l:
                    rouge_ls.append(scorer.score(l, p)['rougeL'].fmeasure)
            rouge_l = np.mean(rouge_ls) if rouge_ls else 0.0

            # METEOR
            meteors = []
            for p, l in zip(pred_norm, label_norm):
                if p and l:
                    meteors.append(meteor_score([l.split()], p.split()))
            meteor = np.mean(meteors) if meteors else 0.0

            # BERTscore
            if valid_preds and valid_labels:
                try:
                    P, R, F1 = bert_score(
                        valid_preds,
                        valid_labels,
                        lang="en",
                        model_type="bert-base-uncased",
                        rescale_with_baseline=False
                    )
                    bertscore_f1 = F1.mean().item()
                except Exception as e:
                    print(f"[Metrics] BERTScore error: {e}")
                    bertscore_f1 = 0.0
            else:
                bertscore_f1 = 0.0

            # MoverScore
            moverscore_val = 0.0
            if moverscore_available and valid_preds and valid_labels:
                try:
                    from moverscore_v2 import get_idf_dict, word_mover_score

                    idf_dict_ref = get_idf_dict(valid_labels)
                    idf_dict_hyp = get_idf_dict(valid_preds)

                    scores = word_mover_score(
                        valid_labels,
                        valid_preds,
                        idf_dict_ref,
                        idf_dict_hyp,
                        stop_words=[],
                        n_gram=1,
                        remove_subwords=True
                    )

                    if scores and len(scores) > 0:
                        moverscore_val = float(np.mean(scores))
                        print(f"[MoverScore] Score: {moverscore_val:.4f}")

                except Exception as e:
                    print(f"[MoverScore] Error: {e}")
                    import traceback
                    traceback.print_exc() # <--- Added this line

            return {
                "rouge_l": float(rouge_l),
                "meteor": float(meteor),
                "bertscore_f1": float(bertscore_f1),
                "moverscore": float(moverscore_val),
            }

        return compute_vqa_metrics

    # Custom Trainer with VRAM clearing before evaluation
    class LoRATrainer(Seq2SeqTrainer):
        """Seq2SeqTrainer subclass that clears VRAM before evaluation."""

        def __init__(self, processor, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.processor = processor

        def evaluate(self, *args, **kwargs):
            print("\nClearing VRAM before evaluation...")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            return super().evaluate(*args, **kwargs)

        def prediction_step(
            self,
            model: torch.nn.Module,
            inputs: dict,
            prediction_loss_only: bool,
            ignore_keys: Optional[list] = None,
        ):
            """Memory-efficient prediction step from GRIT."""
            has_labels = "labels" in inputs
            if has_labels:
                labels = inputs.pop("labels")
            else:
                labels = None

            input_ids = inputs["input_ids"]
            input_len = input_ids.shape[1]

            with torch.no_grad():
                if prediction_loss_only:
                    outputs = model(**inputs)
                    loss = outputs.loss if has_labels else None
                    return (loss, None, None)

                # Generate (full sequence)
                generated_tokens = model.generate(
                    **inputs,
                    max_new_tokens=128,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    pad_token_id=self.processor.tokenizer.pad_token_id,
                    eos_token_id=self.processor.tokenizer.eos_token_id,
                )

                # Slice to generated part only (answer)
                generated_tokens = generated_tokens[:, input_len:]

                # Loss (full inputs for compute_loss)
                loss = None
                if has_labels:
                    inputs_copy = inputs.copy()
                    inputs_copy["labels"] = labels
                    loss = self.compute_loss(model, inputs_copy).detach()

            # Labels processing
            if labels is not None:
                if labels.dim() == 3:
                    labels = torch.argmax(labels, dim=-1)
                elif labels.dim() == 1:
                    batch_size = input_ids.shape[0]
                    labels = labels.view(batch_size, -1)

            return loss, generated_tokens, labels

    trainer = LoRATrainer(
        processor=processor,
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        data_collator=collate_fn,
        compute_metrics=make_compute_metrics(processor),
    )

    print("\n" + "="*80)
    print("STARTING LORA-ONLY TRAINING FOR GEMMA 3-4B-IT MODEL (ABLATION STUDY)")
    print("="*80)
    print(f"Model: {config.model_id}")
    print(f"Dataset: {config.dataset_name}")
    print(f"LoRA rank: {config.lora_rank}")
    print(f"Batch size: {config.batch_size} x {config.gradient_accumulation_steps} = {eff_batch}")
    print(f"Learning rate: {config.learning_rate}")
    print(f"Precision: {config.precision}")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")
    print("="*80 + "\n")

    try:
        print("Beginning training...")
        trainer.train()

        print("\n" + "="*80)
        print("FINAL TEST EVALUATION")
        print("="*80)

        test_results = trainer.evaluate(eval_dataset=tokenized_test_dataset)

        print(f"Test Loss          : {test_results['eval_loss']:.5f}")
        print(f"Test ROUGE-L       : {test_results['eval_rouge_l']:.4f}")
        print(f"Test METEOR        : {test_results['eval_meteor']:.4f}")
        print(f"Test BERTscore F1  : {test_results['eval_bertscore_f1']:.4f}")
        print(f"Test MoverScore    : {test_results['eval_moverscore']:.4f}")

        print("\n" + "="*80)
        print("Saving final model...")
        output_dir = "./final_vqa_lora_only_gemma3_model"
        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)
        print(f"Model saved to {output_dir}")

        print("\n" + "="*80)
        print("TRAINING COMPLETED SUCCESSFULLY")
        print("="*80 + "\n")

    except KeyboardInterrupt:
        print("\n" + "="*80)
        print("TRAINING INTERRUPTED BY USER")
        print("="*80)
        print("Saving checkpoint...")
        try:
            model.save_pretrained("./interrupted_checkpoint_lora_only_gemma3")
            processor.save_pretrained("./interrupted_checkpoint_lora_only_gemma3")
            print("Checkpoint saved to ./interrupted_checkpoint_lora_only_gemma3")
        except Exception as e:
            print(f"Failed to save checkpoint: {e}")

    except Exception as e:
        print("\n" + "="*80)
        print("TRAINING FAILED")
        print("="*80)
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        raise

    finally:
        print("\nCleaning up...")
        _image_cache.clear()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("Cleanup complete")


if __name__ == "__main__":
    main()


Loading processor and model for Gemma 3-4B-IT...


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

... (37 outputs left out)

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[MoverScore] Score: 0.5480


Training Loss,Validation Loss,Step,Rouge L,Meteor,Bertscore F1,Moverscore
No log,26.995014,24,0.238508,0.279342,0.573761,0.547989


Test Loss          : 26.99501
Test ROUGE-L       : 0.2385
Test METEOR        : 0.2793
Test BERTscore F1  : 0.5738
Test MoverScore    : 0.5480

Saving final model...
Model saved to ./final_vqa_lora_only_gemma3_model

TRAINING COMPLETED SUCCESSFULLY


Cleaning up...
Cleanup complete
